In [18]:
# Importing necessary libraries
import numpy as np
import xgboost as xgb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from pathlib import Path

# Load raw data
column_names = ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3'] + [f'sensor_{i}' for i in range(1, 22)]
Data_raw = Path('..') / 'data' / 'raw'
train_df = pd.read_csv(Data_raw / 'train_FD001.txt', sep='\s+', header=None, names=column_names)
test_df = pd.read_csv(Data_raw / 'test_FD001.txt', sep='\s+', header=None, names=column_names)
rul_df = pd.read_csv(Data_raw / 'RUL_FD001.txt', sep='\s+', header=None, names=['RUL'])

# RUL construction
train_df['RUL'] = train_df.groupby('engine_id')['cycle'].transform(max) - train_df['cycle']
train_df['RUL'] = train_df['RUL'].clip(upper=125)

# Informative sensors
informative_sensors = [
    'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7',
    'sensor_9', 'sensor_11', 'sensor_12', 'sensor_14',
    'sensor_15', 'sensor_20', 'sensor_21'
]

# Split by engine_id
train_engines = train_df['engine_id'].unique()
split_idx = int(len(train_engines) * 0.8)
train = train_df[train_df['engine_id'].isin(train_engines[:split_idx])]
val = train_df[train_df['engine_id'].isin(train_engines[split_idx:])]

# Normalizacija
scaler = StandardScaler()
train_scaled = train.copy()
train_scaled[informative_sensors] = scaler.fit_transform(train[informative_sensors])
val_scaled = val.copy()
val_scaled[informative_sensors] = scaler.transform(val[informative_sensors])

# XGBoost
X_train = train_scaled[informative_sensors].values
y_train = train_scaled['RUL'].values
X_val = val_scaled[informative_sensors].values
y_val = val_scaled['RUL'].values

xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(f'XGBoost RMSE: {rmse:.2f}, R2: {r2:.2f}')

# Sekvence za LSTM
def create_sequences(df, sequence_length=30):
    X, y = [], []
    for engine_id in df['engine_id'].unique():
        engine_data = df[df['engine_id'] == engine_id]
        for i in range(len(engine_data) - sequence_length):
            X.append(engine_data[informative_sensors].iloc[i:i+sequence_length].values)
            y.append(engine_data['RUL'].iloc[i+sequence_length])
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sequences(train_scaled)
X_val_seq, y_val_seq = create_sequences(val_scaled)

# LSTM model
model = Sequential([
    LSTM(64, input_shape=(30, 11)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.summary()

history = model.fit(X_train_seq, y_train_seq,
                    validation_data=(X_val_seq, y_val_seq),
                    epochs=20, batch_size=64)

y_pred_lstm = model.predict(X_val_seq).flatten()
rmse_lstm = root_mean_squared_error(y_val_seq, y_pred_lstm)
r2_lstm = r2_score(y_val_seq, y_pred_lstm)
print(f'LSTM RMSE: {rmse_lstm:.2f}, R2: {r2_lstm:.2f}')

XGBoost RMSE: 20.11, R2: 0.76


d:\cmapss-predictive-maintenance\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        19,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,569 (84.25 KB)

 Trainable params: 21,569 (84.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 9s 26ms/step - loss: 4124.6943 - val_loss: 982.0704
Epoch 2/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 450.5409 - val_loss: 315.0770
Epoch 3/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 256.0298 - val_loss: 274.3401
Epoch 4/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 225.5282 - val_loss: 265.2958
Epoch 5/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 216.2018 - val_loss: 189.1323
Epoch 6/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 211.0563 - val_loss: 218.6782
Epoch 7/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 203.0327 - val_loss: 229.6487
Epoch 8/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 198.3438 - val_loss: 204.0630
Epoch 9/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - loss: 194.4806 - val_loss: 207.3204
Epoch 10/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 187.5124 - val_loss: 234.9738
Epoch 11/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 183.6556 - val_loss: 200.